# Unidad 4 · Kimball (Notebook 1)

## Construccion de dim_tiempo y dim_cliente (SQL Server)

Este notebook implementa un mini flujo ETL didactico para crear y poblar dos dimensiones del Data Warehouse:

- `dim_tiempo` (dimension calendario)
- `dim_cliente` (dimension cliente con SCD Tipo 2 simplificado)

Origen: `WideWorldImporters`
Destino: `WideWorldImportersDW2`

Criterios Kimball aplicados:

1. Claves sustitutas (surrogate keys).
2. Atributos descriptivos desnormalizados en dimensiones.
3. Historial de cambios para cliente (SCD Tipo 2).

> Nota: para fines pedagogicos, el flujo esta simplificado respecto al repositorio oficial de Microsoft.

## 1) Dependencias

Si hace falta instalar paquetes, descomentar y ejecutar la siguiente linea:

```python
# %pip install pandas sqlalchemy pyodbc
```

In [1]:
import hashlib
import urllib
from datetime import date, datetime, timedelta

import pandas as pd
import pyodbc
from sqlalchemy import create_engine, text

print('Librerias importadas correctamente')

Librerias importadas correctamente


## 2) Parametros de conexion

En este ejemplo usamos autenticacion SQL.

In [6]:
# Conexion SQL Server
SERVIDOR  = r'localhost\SQLEXPRESS02'
BD_ORIGEN = 'WideWorldImporters'
BD_DESTINO = 'WideWorldImportersDW2'
USUARIO = 'ingesta_reader'
CLAVE = '123456789'

# Rango de la dimension tiempo
ANIO_INICIO = 2010
ANIO_FIN = 2035

# Seleccion automatica de driver
drivers = pyodbc.drivers()
if 'ODBC Driver 18 for SQL Server' in drivers:
    DRIVER = 'ODBC Driver 18 for SQL Server'
elif 'ODBC Driver 17 for SQL Server' in drivers:
    DRIVER = 'ODBC Driver 17 for SQL Server'
else:
    raise RuntimeError('No se encontro ODBC Driver 17/18 para SQL Server')

print(f'Driver: {DRIVER}')
print(f'Origen: {BD_ORIGEN}')
print(f'Destino: {BD_DESTINO}')

Driver: ODBC Driver 18 for SQL Server
Origen: WideWorldImporters
Destino: WideWorldImportersDW2


In [8]:
def construir_engine(base_datos: str):
    conn_str = (
        f'DRIVER={{{DRIVER}}};'
        f'SERVER={SERVIDOR};'
        f'DATABASE={base_datos};'
        f'UID={USUARIO};'
        f'PWD={CLAVE};'
        'TrustServerCertificate=yes;'
    )
    params = urllib.parse.quote_plus(conn_str)
    return create_engine(f'mssql+pyodbc:///?odbc_connect={params}')

engine_origen = construir_engine(BD_ORIGEN)
engine_destino = construir_engine(BD_DESTINO)

with engine_origen.connect() as c1:
    v1 = c1.execute(text('SELECT DB_NAME() AS bd')).mappings().first()
with engine_destino.connect() as c2:
    v2 = c2.execute(text('SELECT DB_NAME() AS bd')).mappings().first()

print(f'Conexion OK origen : {v1["bd"]}')
print(f'Conexion OK destino: {v2["bd"]}')

Conexion OK origen : WideWorldImporters
Conexion OK destino: WideWorldImportersDW2


## 3) DDL del Data Warehouse (tablas en espanol)

Se crea el esquema `dw` y dos dimensiones: `dw.dim_tiempo` y `dw.dim_cliente`.

In [ ]:
sql_ddl = '''
IF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE name = 'dw')
    EXEC('CREATE SCHEMA dw');

IF OBJECT_ID('dw.dim_tiempo', 'U') IS NULL
BEGIN
    CREATE TABLE dw.dim_tiempo (
        id_tiempo                 INT            NOT NULL PRIMARY KEY,   -- YYYYMMDD
        fecha                     DATE           NOT NULL UNIQUE,
        dia_numero               TINYINT        NOT NULL,
        dia_nombre               NVARCHAR(15)   NOT NULL,
        dia_semana_iso           TINYINT        NOT NULL,
        semana_anio              TINYINT        NOT NULL,
        mes_numero               TINYINT        NOT NULL,
        mes_nombre               NVARCHAR(15)   NOT NULL,
        trimestre_numero         TINYINT        NOT NULL,
        trimestre_nombre         NVARCHAR(10)   NOT NULL,
        semestre_numero          TINYINT        NOT NULL,
        anio_numero              SMALLINT       NOT NULL,
        anio_mes                 INT            NOT NULL,
        es_fin_de_semana         BIT            NOT NULL,
        es_dia_habil             BIT            NOT NULL
    );
END;

IF OBJECT_ID('dw.dim_cliente', 'U') IS NULL
BEGIN
    CREATE TABLE dw.dim_cliente (
        id_cliente_sk            INT IDENTITY(1,1) PRIMARY KEY,
        id_cliente_nk            INT            NOT NULL,
        nombre_cliente           NVARCHAR(200)  NOT NULL,
        categoria_cliente        NVARCHAR(100)  NULL,
        grupo_compra             NVARCHAR(100)  NULL,
        ciudad                   NVARCHAR(100)  NULL,
        estado_provincia         NVARCHAR(100)  NULL,
        pais                     NVARCHAR(100)  NULL,
        limite_credito           DECIMAL(18,2)  NULL,
        dias_pago                INT            NULL,
        fecha_inicio_vigencia    DATE           NOT NULL,
        fecha_fin_vigencia       DATE           NOT NULL,
        es_actual                BIT            NOT NULL,
        hash_atributos           CHAR(32)       NOT NULL,
        fecha_carga              DATETIME2      NOT NULL DEFAULT SYSDATETIME()
    );

    CREATE INDEX ix_dim_cliente_nk_actual ON dw.dim_cliente (id_cliente_nk, es_actual);
END;
'''

with engine_destino.begin() as conn:
    conn.execute(text(sql_ddl))

print('DDL ejecutado correctamente')

## 4) Carga de dim_tiempo

Generamos una fila por cada fecha del rango definido.

In [17]:
import requests


def _descargar_feriados_anio(anio: int) -> list[dict]:
    """
    Descarga los feriados/no laborables de Argentina para un año.

    Args:
        anio: Año a consultar en la API.

    Returns:
        Lista de registros (diccionarios). Si el año no existe en la API (404),
        retorna lista vacía para continuar el proceso sin interrumpirlo.
    """
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    url = f"https://api.argentinadatos.com/v1/feriados/{anio}"
    resp = requests.get(url, headers=headers, timeout=20)

    # Si no hay datos para el año, devolvemos vacío (comportamiento tolerante).
    if resp.status_code == 404:
        return []

    resp.raise_for_status()
    data = resp.json()
    return data if isinstance(data, list) else []


def obtener_no_laborables_arg(anio_inicio: int = 2010, fecha_hasta: date | None = None) -> pd.DataFrame:
    """
    Construye un DataFrame con fechas no laborables desde 'anio_inicio' hasta 'fecha_hasta'.

    Args:
        anio_inicio: Año inicial de búsqueda.
        fecha_hasta: Fecha límite inclusiva. Si no se indica, se usa hoy.

    Returns:
        DataFrame con una sola columna: 'fecha' (datetime), sin duplicados y ordenada.
    """
    if fecha_hasta is None:
        fecha_hasta = date.today()

    fechas = []

    # Recorremos año por año para consumir el endpoint anual.
    for anio in range(anio_inicio, fecha_hasta.year + 1):
        datos = _descargar_feriados_anio(anio)
        for r in datos:
            fecha = pd.to_datetime(r.get("fecha")).date()
            if fecha <= fecha_hasta:
                fechas.append(fecha)

    if not fechas:
        return pd.DataFrame(columns=["fecha"])

    df = pd.DataFrame({"fecha": pd.to_datetime(fechas)})
    return df.drop_duplicates().sort_values("fecha").reset_index(drop=True)


# DataFrame base de no laborables consumido por la lógica de dim_tiempo.
df_no_laborables = obtener_no_laborables_arg(2010, date.today())

In [18]:
# Catálogo de nombres de días y meses en español para atributos descriptivos.
dias_es = {
    0: 'Lunes',
    1: 'Martes',
    2: 'Miercoles',
    3: 'Jueves',
    4: 'Viernes',
    5: 'Sabado',
    6: 'Domingo'
}

meses_es = {
    1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril',
    5: 'Mayo', 6: 'Junio', 7: 'Julio', 8: 'Agosto',
    9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'
}


def construir_dim_tiempo(anio_inicio: int, anio_fin: int, no_laborables_df: pd.DataFrame | None = None) -> pd.DataFrame:
    """
    Genera la dimensión tiempo de Kimball con atributos calendario.

    Reglas de negocio aplicadas para es_dia_habil:
      - Domingo: no hábil.
      - Sábado: hábil.
      - Si la fecha aparece en no_laborables_df: no hábil (prioridad máxima).

    Args:
        anio_inicio: Año inicial del rango.
        anio_fin: Año final del rango.
        no_laborables_df: DataFrame con columna 'fecha' de no laborables.

    Returns:
        DataFrame listo para insertar en dw.dim_tiempo.
    """
    fecha_inicio = date(anio_inicio, 1, 1)
    fecha_fin = date(anio_fin, 12, 31)
    fechas = pd.date_range(fecha_inicio, fecha_fin, freq='D')

    # Base diaria de calendario.
    df = pd.DataFrame({'fecha': fechas})

    # Claves y atributos temporales.
    df['id_tiempo'] = df['fecha'].dt.strftime('%Y%m%d').astype(int)
    df['dia_numero'] = df['fecha'].dt.day
    df['dia_nombre'] = df['fecha'].dt.weekday.map(dias_es)
    df['dia_semana_iso'] = df['fecha'].dt.isocalendar().day.astype(int)
    df['semana_anio'] = df['fecha'].dt.isocalendar().week.astype(int)
    df['mes_numero'] = df['fecha'].dt.month
    df['mes_nombre'] = df['mes_numero'].map(meses_es)
    df['trimestre_numero'] = df['fecha'].dt.quarter
    df['trimestre_nombre'] = 'T' + df['trimestre_numero'].astype(str)
    df['semestre_numero'] = ((df['mes_numero'] - 1) // 6) + 1
    df['anio_numero'] = df['fecha'].dt.year
    df['anio_mes'] = (df['anio_numero'] * 100 + df['mes_numero']).astype(int)

    # Bandera de fin de semana (sábado/domingo).
    df['es_fin_de_semana'] = df['fecha'].dt.weekday.isin([5, 6]).astype(int)

    # Set de búsqueda rápida para fechas no laborables.
    if no_laborables_df is not None and not no_laborables_df.empty:
        no_laborables_set = set(pd.to_datetime(no_laborables_df['fecha']).dt.normalize())
    else:
        no_laborables_set = set()

    # Marca proveniente de API de no laborables.
    df['es_no_laborable_api'] = df['fecha'].dt.normalize().isin(no_laborables_set)

    # Regla final de hábil/no hábil según requerimiento funcional.
    df['es_dia_habil'] = (
        (df['fecha'].dt.weekday != 6) &
        (~df['es_no_laborable_api'])
    ).astype(int)

    # Orden de columnas destino.
    columnas = [
        'id_tiempo', 'fecha', 'dia_numero', 'dia_nombre', 'dia_semana_iso',
        'semana_anio', 'mes_numero', 'mes_nombre', 'trimestre_numero',
        'trimestre_nombre', 'semestre_numero', 'anio_numero', 'anio_mes',
        'es_fin_de_semana', 'es_dia_habil'
    ]
    return df[columnas]


# Construcción final de la dimensión tiempo usando no laborables descargados.
df_tiempo = construir_dim_tiempo(ANIO_INICIO, ANIO_FIN, df_no_laborables)
print(f'Filas generadas para dim_tiempo: {len(df_tiempo)}')
df_tiempo.head()

Filas generadas para dim_tiempo: 9496


,id_tiempo,fecha,dia_numero,dia_nombre,dia_semana_iso,semana_anio,mes_numero,mes_nombre,trimestre_numero,trimestre_nombre,semestre_numero,anio_numero,anio_mes,es_fin_de_semana,es_dia_habil
0,20100101,2010-01-01,1,Viernes,5,53,1,Enero,1,T1,1,2010,201001,0,1
1,20100102,2010-01-02,2,Sabado,6,53,1,Enero,1,T1,1,2010,201001,1,1
2,20100103,2010-01-03,3,Domingo,7,53,1,Enero,1,T1,1,2010,201001,1,0
3,20100104,2010-01-04,4,Lunes,1,1,1,Enero,1,T1,1,2010,201001,0,1
4,20100105,2010-01-05,5,Martes,2,1,1,Enero,1,T1,1,2010,201001,0,1


In [ ]:
with engine_destino.connect() as conn:
    existentes = pd.read_sql('SELECT id_tiempo FROM dw.dim_tiempo', conn)

if existentes.empty:
    df_nuevas = df_tiempo.copy()
else:
    df_nuevas = df_tiempo[~df_tiempo['id_tiempo'].isin(existentes['id_tiempo'])].copy()

if not df_nuevas.empty:
    df_nuevas.to_sql('dim_tiempo', engine_destino, schema='dw', if_exists='append', index=False)

print(f'Registros nuevos insertados en dim_tiempo: {len(df_nuevas)}')

## 5) Extraccion de clientes desde WideWorldImporters

Consulta simplificada para construir la dimension cliente en espanol.

### Explicacion de la extraccion de clientes

La extraccion de `dim_cliente` toma como fuente principal `Sales.Customers` y enriquece cada cliente con atributos descriptivos usando joins a tablas maestras de WideWorldImporters.

**Objetivo:** construir una vista desnormalizada del cliente para analitica (enfoque Kimball).

**Tablas usadas:**
- `Sales.Customers` (`c`): clave natural del cliente, nombre, limite de credito y dias de pago.
- `Sales.CustomerCategories` (`cc`): categoria del cliente.
- `Sales.BuyingGroups` (`bg`): grupo de compra.
- `Application.Cities` (`ci`): ciudad.
- `Application.StateProvinces` (`sp`): estado/provincia.
- `Application.Countries` (`co`): pais.

**Tipo de joins:** `LEFT JOIN`.

Se usan `LEFT JOIN` para no perder clientes aunque falte algun dato descriptivo en tablas relacionadas. Luego el resultado se carga en `df_clientes_src` y se utiliza en el paso SCD Tipo 2 para insertar o versionar registros en `dw.dim_cliente`.

In [ ]:
sql_clientes_origen = '''
SELECT
    c.CustomerID                                  AS id_cliente_nk,
    c.CustomerName                                AS nombre_cliente,
    cc.CustomerCategoryName                       AS categoria_cliente,
    bg.BuyingGroupName                            AS grupo_compra,
    ci.CityName                                   AS ciudad,
    sp.StateProvinceName                          AS estado_provincia,
    co.CountryName                                AS pais,
    c.CreditLimit                                 AS limite_credito,
    c.PaymentDays                                 AS dias_pago
FROM Sales.Customers c
LEFT JOIN Sales.CustomerCategories cc
    ON c.CustomerCategoryID = cc.CustomerCategoryID
LEFT JOIN Sales.BuyingGroups bg
    ON c.BuyingGroupID = bg.BuyingGroupID
LEFT JOIN Application.Cities ci
    ON c.DeliveryCityID = ci.CityID
LEFT JOIN Application.StateProvinces sp
    ON ci.StateProvinceID = sp.StateProvinceID
LEFT JOIN Application.Countries co
    ON sp.CountryID = co.CountryID;
'''

with engine_origen.connect() as conn:
    df_clientes_src = pd.read_sql(sql_clientes_origen, conn)

print(f'Clientes extraidos desde origen: {len(df_clientes_src)}')
df_clientes_src.head()

## 6) Carga SCD Tipo 2 simplificada de dim_cliente

Regla aplicada:

- Si el cliente no existe: insertar version actual.
- Si existe y cambio algun atributo descriptivo: cerrar version anterior e insertar nueva version.
- Si no hay cambios: no hacer nada.

In [ ]:
COLUMNAS_HASH = [
    'nombre_cliente', 'categoria_cliente', 'grupo_compra',
    'ciudad', 'estado_provincia', 'pais', 'limite_credito', 'dias_pago'
]

def hash_fila(row, cols):
    valores = []
    for c in cols:
        v = row[c]
        if pd.isna(v):
            valores.append('')
        else:
            valores.append(str(v).strip())
    cadena = '|'.join(valores)
    return hashlib.md5(cadena.encode('utf-8')).hexdigest()

hoy = date.today()
fin_abierto = date(9999, 12, 31)

df_clientes_src = df_clientes_src.copy()
df_clientes_src['hash_atributos'] = df_clientes_src.apply(lambda r: hash_fila(r, COLUMNAS_HASH), axis=1)

sql_actual = '''
SELECT
    id_cliente_sk,
    id_cliente_nk,
    hash_atributos
FROM dw.dim_cliente
WHERE es_actual = 1;
'''

with engine_destino.connect() as conn:
    df_actual = pd.read_sql(sql_actual, conn)

if df_actual.empty:
    df_insertar = df_clientes_src.copy()
    df_insertar['fecha_inicio_vigencia'] = hoy
    df_insertar['fecha_fin_vigencia'] = fin_abierto
    df_insertar['es_actual'] = 1

    columnas_insert = [
        'id_cliente_nk', 'nombre_cliente', 'categoria_cliente', 'grupo_compra',
        'ciudad', 'estado_provincia', 'pais', 'limite_credito', 'dias_pago',
        'fecha_inicio_vigencia', 'fecha_fin_vigencia', 'es_actual', 'hash_atributos'
    ]
    df_insertar[columnas_insert].to_sql('dim_cliente', engine_destino, schema='dw', if_exists='append', index=False)

    print(f'Insercion inicial completada: {len(df_insertar)} filas')
else:
    df_merge = df_clientes_src.merge(
        df_actual,
        on='id_cliente_nk',
        how='left',
        suffixes=('', '_actual')
    )

    nuevos = df_merge[df_merge['id_cliente_sk'].isna()].copy()
    cambiados = df_merge[
        (~df_merge['id_cliente_sk'].isna()) &
        (df_merge['hash_atributos'] != df_merge['hash_atributos_actual'])
    ].copy()

    with engine_destino.begin() as conn:
        if not cambiados.empty:
            ids_sk = tuple(cambiados['id_cliente_sk'].astype(int).tolist())
            if len(ids_sk) == 1:
                where_in = f'({ids_sk[0]})'
            else:
                where_in = str(ids_sk)

            sql_cierre = f'''
            UPDATE dw.dim_cliente
            SET
                fecha_fin_vigencia = :fecha_fin,
                es_actual = 0
            WHERE id_cliente_sk IN {where_in};
            '''
            conn.execute(text(sql_cierre), {'fecha_fin': hoy - timedelta(days=1)})

        inserciones = pd.concat([nuevos, cambiados], ignore_index=True)
        if not inserciones.empty:
            inserciones = inserciones[ [
                'id_cliente_nk', 'nombre_cliente', 'categoria_cliente', 'grupo_compra',
                'ciudad', 'estado_provincia', 'pais', 'limite_credito', 'dias_pago', 'hash_atributos'
            ] ].copy()
            inserciones['fecha_inicio_vigencia'] = hoy
            inserciones['fecha_fin_vigencia'] = fin_abierto
            inserciones['es_actual'] = 1

            inserciones.to_sql('dim_cliente', conn, schema='dw', if_exists='append', index=False)

    print(f'Nuevos clientes insertados : {len(nuevos)}')
    print(f'Clientes versionados SCD2  : {len(cambiados)}')

## 7) Validaciones finales

Consultas para revisar rapidamente el resultado de la carga.

In [ ]:
sql_validacion = '''
SELECT 'dim_tiempo' AS tabla, COUNT(*) AS filas FROM dw.dim_tiempo
UNION ALL
SELECT 'dim_cliente' AS tabla, COUNT(*) AS filas FROM dw.dim_cliente;
'''

with engine_destino.connect() as conn:
    df_resumen = pd.read_sql(sql_validacion, conn)

df_resumen

In [ ]:
sql_muestra = '''
SELECT TOP 20
    id_cliente_sk,
    id_cliente_nk,
    nombre_cliente,
    categoria_cliente,
    ciudad,
    pais,
    fecha_inicio_vigencia,
    fecha_fin_vigencia,
    es_actual
FROM dw.dim_cliente
ORDER BY id_cliente_nk, fecha_inicio_vigencia DESC;
'''

with engine_destino.connect() as conn:
    df_muestra = pd.read_sql(sql_muestra, conn)

df_muestra

## 8) Proximo paso (Notebook 2)

Con estas dimensiones listas, el siguiente notebook puede enfocarse en:

1. `dim_producto`
2. `fact_ventas` con grano: una linea por detalle de factura
3. Relacion por claves sustitutas (`id_tiempo`, `id_cliente_sk`, `id_producto_sk`)